# E038 — Ensemble UBM: Multiple Seeds Averaging

**Motivation:** GMM training is stochastic (k-means initialization). Ensemble of multiple UBMs (different seeds) with score averaging may reduce variance and improve robustness.

**Hypothesis:** Ensemble of 3-5 UBMs (different random seeds) with averaged LLR scores will reduce EER variance across folds and improve mean EER.

**Approach:**
1. Train N UBMs with different seeds (N=3, 5)
2. MAP adapt each UBM to target
3. Score test sample with all N models
4. Average LLR scores
5. Compare to single UBM (E037 tied)

**Configs:**
- `single`: E037 baseline (tied cov, seed=67)
- `ens3`: Ensemble of 3 UBMs (seeds 67, 68, 69)
- `ens5`: Ensemble of 5 UBMs (seeds 67-71)

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

BASE_SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E037_REF = {'mean_eer': 0.69, 'std_eer': 0.98}

In [ ]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def train_ubm_map_ensemble(X_target, X_nontarget, seeds):
    """Train ensemble of UBM-MAP models."""
    models = []
    for seed in seeds:
        ubm = GaussianMixture(
            n_components=UBM_COMPONENTS,
            covariance_type='tied',
            max_iter=200,
            random_state=seed
        ).fit(X_nontarget)
        
        log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
        log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
        resp = np.exp(log_resp)
        n_k = resp.sum(axis=0)
        mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
        alpha = n_k / (n_k + MAP_R)
        
        adapted = copy.deepcopy(ubm)
        adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
        
        models.append((ubm, adapted))
    
    return models

def score_ensemble(wav_path, models):
    """Score with ensemble and average LLR."""
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat = _extract_lpcc(y, sr)
    
    scores = []
    for ubm, adapted in models:
        score = adapted.score(feat) - ubm.score(feat)
        scores.append(score)
    
    return np.mean(scores)

def extract_features(df, data_dir, seed):
    """Extract LPCC features with pitch augmentation for target."""
    rng = np.random.default_rng(seed)
    X_target, X_nontarget = [], []
    
    for _, row in df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if row["label"] == 1:
            wavs.append(_aug_pitch(y_wav, sr, rng))
        
        for y_aug in wavs:
            feat = _extract_lpcc(y_aug, sr)
            if row["label"] == 1:
                X_target.append(feat)
            else:
                X_nontarget.append(feat)
    
    return np.vstack(X_target), np.vstack(X_nontarget)

print('Model functions defined')

## 2. Cross-validation

In [ ]:
configs = {
    'single': [BASE_SEED],
    'ens3': [BASE_SEED, BASE_SEED+1, BASE_SEED+2],
    'ens5': [BASE_SEED, BASE_SEED+1, BASE_SEED+2, BASE_SEED+3, BASE_SEED+4],
}

results = {}

for name, seeds in configs.items():
    print(f"\n=== {name} (seeds={seeds}) ===")
    fold_eers = []
    
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=BASE_SEED):
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        
        X_target, X_nontarget = extract_features(train_df, DATA, BASE_SEED)
        models = train_ubm_map_ensemble(X_target, X_nontarget, seeds)
        
        scores = []
        for _, row in val_df.iterrows():
            score = score_ensemble(_find_wav(row["stem"], DATA), models)
            scores.append(score)
        
        scores = np.array(scores)
        eer, _ = compute_eer(scores[y_all[val_idx] == 1], scores[y_all[val_idx] == 0])
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    
    results[name] = {
        'fold_eers': fold_eers,
        'mean': np.mean(fold_eers),
        'std': np.std(fold_eers),
    }
    print(f"  Mean ± Std: {np.mean(fold_eers):.2f} ± {np.std(fold_eers):.2f}%")

print("\n=== Summary ===")
for name, res in results.items():
    print(f"{name:12s}: {res['mean']:5.2f} ± {res['std']:5.2f}%  (improvement: {results['single']['mean'] - res['mean']:+.2f}pp)")

## 3. Conclusion

Ensemble effect: [TBD]
Decision: [Adopt/Reject] — trade-off: inference cost vs accuracy